In [1]:
import pandas as pd
import numpy as np
import polars as pl
from sklearn.metrics import mutual_info_score
from sklearn.model_selection import train_test_split
from IPython.display import display
from sklearn.metrics import mutual_info_score
from datetime import datetime
import os

In [ ]:
mes_proceso="2026-06-01"

In [2]:
df=pl.read_parquet("data/deuda_old.parquet").to_pandas()

In [3]:
df.columns=df.columns.str.lower().str.replace(" ","_")
categorical_columns=list(df.dtypes[df.dtypes=="str"].index)
for c in categorical_columns:
    df[c]=df[c].str.lower().str.replace(" ","_")

df.head()

,nombre_entidad,situacion,provincia,sexo,tipo_persona,prestamos,cantidad_deudores,fecha,situacion_mora,descripcion,rango_etario
0,tarjeta_naranja_s.a.u.,1,santa_fe,m,física,2058658.0,3821,2026-06-01,normal/bajo_riesgo,sin_actividad,<25
1,banco_de_galicia_y_buenos_aires_s.a.,1,buenos_aires,m,física,518939623.0,102206,2026-06-01,normal/bajo_riesgo,sin_actividad,45-54
2,recupero_de_activos_fideicomiso_financiero,5,san_juan,f,física,757038.0,1382,2026-06-01,en_mora,sin_actividad,65<
3,finanzas_digitales_sas,2,catamarca,m,física,4245.0,81,2026-06-01,normal/bajo_riesgo,sin_actividad,25-34
4,naranja_digital_compañía_financiera_s.a.u.,2,salta,f,física,580561.0,647,2026-06-01,normal/bajo_riesgo,sin_actividad,35-44


In [4]:
df.isnull().sum()

nombre_entidad       0
situacion            0
provincia            0
sexo                 0
tipo_persona         0
prestamos            0
cantidad_deudores    0
fecha                0
situacion_mora       0
descripcion          0
rango_etario         0
dtype: int64

In [5]:
df.situacion_mora=(df.situacion_mora=='en_mora').astype(int)
df.head()

,nombre_entidad,situacion,provincia,sexo,tipo_persona,prestamos,cantidad_deudores,fecha,situacion_mora,descripcion,rango_etario
0,tarjeta_naranja_s.a.u.,1,santa_fe,m,física,2058658.0,3821,2026-06-01,0,sin_actividad,<25
1,banco_de_galicia_y_buenos_aires_s.a.,1,buenos_aires,m,física,518939623.0,102206,2026-06-01,0,sin_actividad,45-54
2,recupero_de_activos_fideicomiso_financiero,5,san_juan,f,física,757038.0,1382,2026-06-01,1,sin_actividad,65<
3,finanzas_digitales_sas,2,catamarca,m,física,4245.0,81,2026-06-01,0,sin_actividad,25-34
4,naranja_digital_compañía_financiera_s.a.u.,2,salta,f,física,580561.0,647,2026-06-01,0,sin_actividad,35-44


In [6]:
totales_por_situacion=df.groupby('situacion_mora')['cantidad_deudores'].sum()
proporciones=totales_por_situacion/totales_por_situacion.sum()
tasa_global_mora = proporciones[1]
tasa_global_mora

np.float64(0.24661270463074475)

In [7]:
df_mujeres = df[df['sexo'] == 'f']
mora_female = (df_mujeres['situacion_mora'] * df_mujeres['cantidad_deudores']).sum() / df_mujeres['cantidad_deudores'].sum()

df_varones = df[df['sexo'] == 'm']
mora_varones = (df_varones['situacion_mora'] * df_varones['cantidad_deudores']).sum() / df_varones['cantidad_deudores'].sum()

print(mora_female,mora_varones)

0.23822240403013212 0.25837722559752446


In [8]:
df['personas_en_mora']=df['situacion_mora']*df['cantidad_deudores']
df_group=df.groupby('sexo')[['personas_en_mora','cantidad_deudores']].sum()

df_group['mean']=df_group['personas_en_mora']/df_group['cantidad_deudores']
df_group['count']=df_group['cantidad_deudores']

df_group['diff'] = df_group['mean'] - tasa_global_mora 
df_group['risk'] = df_group['mean'] / tasa_global_mora 

df_group = df_group[['mean', 'count', 'diff', 'risk']]

print(df_group)

             mean     count      diff      risk
sexo                                           
empresa  0.144670    522183 -0.101943  0.586627
f        0.238222  20876735 -0.008390  0.965978
m        0.258377  19414393  0.011765  1.047704
x        0.239264       815 -0.007349  0.970201


In [9]:
df.columns

Index(['nombre_entidad', 'situacion', 'provincia', 'sexo', 'tipo_persona',
       'prestamos', 'cantidad_deudores', 'fecha', 'situacion_mora',
       'descripcion', 'rango_etario', 'personas_en_mora'],
      dtype='str')

In [ ]:
categorical=['provincia', 'sexo', 'tipo_persona','rango_etario','descripcion']
numeric=['personas_en_mora','prestamos', 'cantidad_deudores']
provincias=['santa_fe',        'buenos_aires',            'san_juan',
           'catamarca',               'salta',           'río_negro',
            'misiones',            'san_luis',             'córdoba',
             'tucumán',             'mendoza',               'chaco',
                'caba',          'entre_ríos',            'la_pampa',
             'neuquén',              'chubut',          'santa_cruz',
               'jujuy',    'tierra_del_fuego',          'corrientes',
 'santiago_del_estero',            'la_rioja',             'formosa',
     'sin_identificar']

In [42]:
df[categorical].nunique()

provincia         25
sexo               4
tipo_persona       3
rango_etario       8
descripcion     1061
dtype: int64

In [68]:
resultados_globales = []

for c in categorical:
    print(c)
    df_group=df.groupby(c)[['personas_en_mora','cantidad_deudores','prestamos']].sum()

    df_group['mean']=df_group['personas_en_mora']/df_group['cantidad_deudores']
    df_group['count']=df_group['cantidad_deudores']
    df_group['prestamos']=df_group['prestamos']
    df_group['diff'] = df_group['mean'] - tasa_global_mora 
    df_group['risk'] = df_group['mean'] / tasa_global_mora 

    df_group = df_group[['mean', 'count','prestamos', 'diff', 'risk']]

    df_group.reset_index(inplace=True)
        
    df_group.rename(columns={c: 'caracteristica'}, inplace=True)

    df_group['columna'] = c
    df_group['provincia']='global'

    resultados_globales.append(df_group)
    
df_metricas_global = pd.concat(resultados_globales, ignore_index=True)

df_metricas_global = df_metricas_global[['provincia','columna', 'caracteristica', 'mean', 'count','prestamos', 'diff', 'risk']]


provincia
sexo
tipo_persona
rango_etario
descripcion


In [90]:
resultados_globales_provincia = []

for p in provincias:
    print(f"Procesando provincia: {p}...")
    
    df_prov = df[df['provincia'] == p].copy()
    
    if df_prov.empty:
        continue

    for c in categorical:
        df_group = df_prov.groupby(c)[['personas_en_mora', 'cantidad_deudores','prestamos']].sum()

        df_group['mean'] = df_group['personas_en_mora'] / df_group['cantidad_deudores']
        df_group['count'] = df_group['cantidad_deudores']
        df_group['prestamos']=df_group['prestamos']
        df_group['diff'] = df_group['mean'] - tasa_global_mora 
        df_group['risk'] = df_group['mean'] / tasa_global_mora 
        
        df_group = df_group[['mean', 'count','prestamos', 'diff', 'risk']].copy()
        
        df_group.reset_index(inplace=True)
        
        df_group.rename(columns={c: 'caracteristica'}, inplace=True)
        
        df_group['provincia'] = p
        df_group['columna'] = c
        
        resultados_globales_provincia.append(df_group)

df_metricas_provincia = pd.concat(resultados_globales_provincia, ignore_index=True)

df_metricas_provincia = df_metricas_provincia[['provincia', 'columna', 'caracteristica', 'mean', 'count','prestamos', 'diff', 'risk']]

df_metricas_total=pd.concat([df_metricas_global,df_metricas_provincia],ignore_index=True)
df_metricas_total['fecha']=pd.to_datetime("2026-06")

ruta_archivo = 'data/metricas_totales.parquet'

if os.path.exists(ruta_archivo):
    print("Archivo histórico encontrado. Leyendo datos anteriores...")
    df_historico = pl.read_parquet(ruta_archivo).to_pandas()
    
    fecha_dt = pd.to_datetime(mes_proceso)
    df_historico = df_historico[df_historico['fecha'] != fecha_dt]
    df_final = pd.concat([df_historico, df_metricas_total], ignore_index=True)

else:
    print("No se encontró archivo histórico. Se creará uno nuevo.")
    df_final = df_metricas_total

df_final.to_parquet(ruta_archivo, index=False)
print(f"Proceso finalizado. El dataset ahora tiene {len(df_final)} filas.")

Procesando provincia: santa_fe...
Procesando provincia: buenos_aires...
Procesando provincia: san_juan...
Procesando provincia: catamarca...
Procesando provincia: salta...
Procesando provincia: río_negro...
Procesando provincia: misiones...
Procesando provincia: san_luis...
Procesando provincia: córdoba...
Procesando provincia: tucumán...
Procesando provincia: mendoza...
Procesando provincia: chaco...
Procesando provincia: caba...
Procesando provincia: entre_ríos...
Procesando provincia: la_pampa...
Procesando provincia: neuquén...
Procesando provincia: chubut...
Procesando provincia: santa_cruz...
Procesando provincia: jujuy...
Procesando provincia: tierra_del_fuego...
Procesando provincia: corrientes...
Procesando provincia: santiago_del_estero...
Procesando provincia: la_rioja...
Procesando provincia: formosa...
Procesando provincia: sin_identificar...
No se encontró archivo histórico. Se creará uno nuevo.
Proceso finalizado. El dataset ahora tiene 13700 filas.


In [ ]:
categorical_cruzado=['descripcion', 'sexo', 'rango_etario', 'tipo_persona']
columnas_agrupacion = ['provincia'] + categorical_cruzado 

df_cruzado = df.groupby(columnas_agrupacion)[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()

df_cruzado['mean'] = df_cruzado['personas_en_mora'] / df_cruzado['cantidad_deudores']
df_cruzado['count'] = df_cruzado['cantidad_deudores']
df_cruzado['prestamos']=df_cruzado['prestamos']
df_cruzado['diff'] = df_cruzado['mean'] - tasa_global_mora 
df_cruzado['risk'] = df_cruzado['mean'] / tasa_global_mora 

df_cruzado = df_cruzado[columnas_agrupacion + ['mean', 'count', 'prestamos', 'diff', 'risk']]

df_global_cruzado = df.groupby(categorical_cruzado)[['personas_en_mora', 'cantidad_deudores', 'prestamos']].sum().reset_index()

df_global_cruzado['mean'] = df_global_cruzado['personas_en_mora'] / df_global_cruzado['cantidad_deudores']
df_global_cruzado['count'] = df_global_cruzado['cantidad_deudores']
df_global_cruzado['prestamos']=df_global_cruzado['prestamos']
df_global_cruzado['diff'] = df_global_cruzado['mean'] - tasa_global_mora 
df_global_cruzado['risk'] = df_global_cruzado['mean'] / tasa_global_mora 

df_global_cruzado['provincia'] = 'global'
df_global_cruzado = df_global_cruzado[['provincia']+categorical_cruzado + ['mean', 'count', 'prestamos', 'diff', 'risk']]

df_metricas_total_cruzado = pd.concat([df_global_cruzado, df_cruzado], ignore_index=True)
df_metricas_total_cruzado['fecha'] = pd.to_datetime("2026-06")

df_metricas_total_cruzado['fecha'] = pd.to_datetime(mes_proceso)

ruta_archivo_cruzado = 'data/metricas_totales_cruzadas.parquet' 

if os.path.exists(ruta_archivo_cruzado):
    print("Archivo histórico encontrado. Actualizando datos...")
    df_historico_cruzado = pd.read_parquet(ruta_archivo_cruzado)
    
    fecha_dt_cruzado = pd.to_datetime(mes_proceso)
    df_historico_cruzado = df_historico_cruzado[df_historico_cruzado['fecha'] != fecha_dt_cruzado]
    
    df_final_cruzado = pd.concat([df_historico_cruzado, df_metricas_total_cruzado], ignore_index=True)
else:
    print("No se encontró archivo histórico. Se creará uno nuevo.")
    df_final_cruzado = df_metricas_total_cruzado

df_final_cruzado.to_parquet(ruta_archivo_cruzado, index=False)

print(f"Proceso finalizado para {mes_proceso}.")
print(f"El dataset histórico ahora tiene {len(df_metricas_total_cruzado)} filas.")

No se encontró archivo histórico. Se creará uno nuevo.
Proceso finalizado para 2026-06-01.
El dataset histórico ahora tiene 13776 filas.


In [51]:
variables_predictoras = ['nombre_entidad','provincia', 'sexo', 'tipo_persona','rango_etario','descripcion']
target = 'situacion_mora'
frecuencia = 'cantidad_deudores'

resultados_mi = {}

for var in variables_predictoras:
    contingency = df.pivot_table(
        index=var, 
        columns=target, 
        values=frecuencia, 
        aggfunc='sum', 
        fill_value=0
    ).values
    
    score = mutual_info_score(None, None, contingency=contingency)
    resultados_mi[var] = score

df_importancia = pd.DataFrame(
    list(resultados_mi.items()), 
    columns=['variable', 'mutual_information']
).sort_values(by='mutual_information', ascending=False)

print(df_importancia)

         variable  mutual_information
0  nombre_entidad            0.091411
4    rango_etario            0.013459
1       provincia            0.004143
5     descripcion            0.000734
2            sexo            0.000675
3    tipo_persona            0.000529
